In [4]:
## This forcefully terminates any hidden Java/Spark processes hoarding your RAM
!pkill -f java
print("All zombie processes cleared. Memory freed!")

All zombie processes cleared. Memory freed!


In [2]:
## 1. Uninstall the "too new" version of PySpark
!pip uninstall -y pyspark

## 2. Install the rock-solid version that matches Java 11
!pip install pyspark==3.5.1

Found existing installation: pyspark 4.1.1
Uninstalling pyspark-4.1.1:
  Successfully uninstalled pyspark-4.1.1
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 700.0 kB/s  0:00:320:00:010:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488549 sha256=df056e667268d9e21361d5ff11c5703eb0a01fd79d22a5e01d75760b61a21800
  Stored in directory: /home/6943f0ec-c509-4fba-a05e-614590f5d507/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark

In [2]:
## ============================================================================
## 1. INFRASTRUCTURE SETUP (SIDE-LOADING JAVA)
## ============================================================================

import os
from pyspark.sql import SparkSession

# 1. Force the Environment Variables to be active right now
current_dir = os.getcwd()
jdk_folder = "jdk-11.0.31+11" # Using the exact folder from your extraction
jdk_path = os.path.join(current_dir, jdk_folder)

os.environ["JAVA_HOME"] = jdk_path
os.environ["PATH"] = f"{jdk_path}/bin:" + os.environ["PATH"]
# This specific variable tells PySpark exactly which Python to use
os.environ["PYSPARK_PYTHON"] = "python3" 

print("Environment set. Booting Spark Engine...")

## ============================================================================
## 2. INITIALIZE SPARK (THE ENGINE)
## ============================================================================
## We use 1GB of memory to stay within the JupyterHub container limits
try:
    spark = SparkSession.builder \
        .appName("Healthcare_Medallion_Pipeline") \
        .config("spark.driver.memory", "1g") \
        .config("spark.executor.memory", "1g") \
        .getOrCreate()
    
    print(" Spark Session Started Successfully!")
except Exception as e:
    print(f" Spark failed to start: {e}")

Environment set. Booting Spark Engine...


26/05/06 23:58:09 WARN Utils: Your hostname, blue-nbjupyterhub3 resolves to a loopback address: 127.0.0.1; using 10.0.0.71 instead (on interface ens5)
26/05/06 23:58:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 23:58:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


 Spark Session Started Successfully!


In [3]:
## ============================================================================
## 3. EXTRACTION (THE "BRONZE" LAYER)
## =================-----------------------------------------------------------
## The Goal: Pull raw data into the system without changing it.
## Read the raw CSV file exactly as it comes from CMS.gov.
## 'header=True' tells Spark the first row contains column names, not data.
## 'inferSchema=True' forces Spark to scan the file and automatically 
## figure out if a column is text, an integer, or a decimal.
## ----------------------------------------------------------------------------
bronze_df = spark.read.csv("medicare_data.csv", header=True, inferSchema=True)

## --- DATA VALIDATION 1 (PRE-CLEANING) ---
## We count the rows and columns immediately upon extraction to prove 
## to ourselves (and our audit logs) that the data was ingested completely.
raw_row_count = bronze_df.count()
raw_col_count = len(bronze_df.columns)

print("=== BRONZE LAYER VALIDATION ===")
print(f"Total Raw Rows Ingested: {raw_row_count}")
print(f"Total Columns Ingested: {raw_col_count}")
print("===============================\n")

=== BRONZE LAYER VALIDATION ===
Total Raw Rows Ingested: 183766
Total Columns Ingested: 84



In [7]:
## -----------------------------------------
## STEP 4: TRANSFORMATION (THE "SILVER" LAYER)
## -----------------------------------------
## The Goal: Clean the data, rename columns, and drop the 79 unused columns.
## Filter: We drop any row that doesn't have a Provider ID (NPI) or a City,
## because those records are useless for our final geographic analysis.
# 1. Import the missing 'col' function!
from pyspark.sql.functions import col
# 2. Clean the data using the correct government column names
silver_df = bronze_df.filter(
    col("Prscrbr_NPI").isNotNull() & 
    col("Prscrbr_City").isNotNull()
).select(
    col("Prscrbr_NPI").alias("provider_id"),
    col("Prscrbr_Last_Org_Name").alias("provider_name"),
    col("Prscrbr_City").alias("city"),
    col("Tot_Clms").cast("int").alias("total_claims"),
    col("Tot_Drug_Cst").cast("double").alias("total_cost")
).dropDuplicates()

# Cache the slim version to RAM to save memory
silver_df.cache()

# 3. Audit the cleaning process
clean_row_count = silver_df.count()
print(f"Silver Layer Created! Filtered down to {clean_row_count:,} valid rows.")





[Stage 9:=====================================================> (194 + 4) / 200]

Silver Layer Created! Filtered down to 183,766 valid rows.


In [8]:
## --- DATA VALIDATION 2 (POST-CLEANING) ---
## Check exactly how many bad/corrupt records we caught and threw away.
## This is a crucial metric for Data Engineers to monitor source data quality.
clean_row_count = silver_df.count()
rows_removed = raw_row_count - clean_row_count

print("=== SILVER LAYER VALIDATION ===")
print(f"Bad Rows Removed (Nulls/Duplicates): {rows_removed}")
print(f"Final Clean Rows Ready for Analysis: {clean_row_count}")
print("===============================\n")

## --- LOADING 1 (SAVE SILVER DATA) ---
## Save this clean data as a Parquet file locally. Parquet is highly compressed 
## and much faster to read than CSVs. This acts as our secure "Clean Room" database.
silver_df.write.mode("overwrite").parquet("healthcare_silver.parquet")

=== SILVER LAYER VALIDATION ===
Bad Rows Removed (Nulls/Duplicates): 0
Final Clean Rows Ready for Analysis: 183766



In [10]:
## ============================================================================
## STEP 5: BUSINESS LOGIC (THE "GOLD" LAYER)
## ============================================================================
# CRITICAL IMPORT: Forces Python to use Spark's heavy-duty math functions
from pyspark.sql.functions import col, sum, count

## Read back from the optimized Silver Parquet files we just created.
clean_data = spark.read.parquet("healthcare_silver.parquet")

## Aggregation: Group the data by city (like a giant pivot table).
gold_df = clean_data.groupBy("city") \
    .agg(
        sum("total_cost").alias("total_city_spend"), 
        count("provider_id").alias("unique_providers_count")
    ) \
    .orderBy(col("total_city_spend").desc()) ## Sort most expensive to the top

## --- LOADING 2 (SAVE GOLD DATA) ---
## Save the final analytical report
gold_df.write.mode("overwrite").parquet("healthcare_gold_report.parquet")

print("Gold Layer Created Successfully. Top 5 Cities by Spend:")
gold_df.show(5)

## Shut down the Spark engine to give memory back to the JupyterHub environment.
spark.stop()

Gold Layer Created Successfully. Top 5 Cities by Spend:


+------------+--------------------+----------------------+
|        city|    total_city_spend|unique_providers_count|
+------------+--------------------+----------------------+
|    New York| 5.927755036899998E8|                  3086|
|    Brooklyn|3.9115309544999987E8|                  1353|
|     Houston|      3.6453737117E8|                  1862|
|Philadelphia|3.2815749860999995E8|                  1582|
| Los Angeles|       3.204753565E8|                  1521|
+------------+--------------------+----------------------+
only showing top 5 rows



In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import format_number, col

# 1. Turn the engine back on quickly
spark = SparkSession.builder.appName("Gold_Report_Viewer").getOrCreate()

# 2. Read the final Gold Parquet file you just saved
final_report = spark.read.parquet("healthcare_gold_report.parquet")

print("--- TOP 5 CITIES BY SPEND (FORMATTED) ---")

# 3. Format the massive numbers to have commas and 2 decimal places
final_report.select(
    col("city"),
    format_number("total_city_spend", 0).alias("Total_Spend_($)"),
    col("unique_providers_count").alias("Doctor_Count")
).show(5, truncate=False)

spark.stop()

--- TOP 5 CITIES BY SPEND (FORMATTED) ---
+------------+---------------+------------+
|city        |Total_Spend_($)|Doctor_Count|
+------------+---------------+------------+
|New York    |592,775,504    |3086        |
|Brooklyn    |391,153,095    |1353        |
|Houston     |364,537,371    |1862        |
|Philadelphia|328,157,499    |1582        |
|Los Angeles |320,475,356    |1521        |
+------------+---------------+------------+
only showing top 5 rows

